# AEM4L5 · Notebook 3 — Profiling, CPU/IO-bound y asyncio con LangChain

**Clase:** Arquitecturas avanzadas de adaptación.

Diagnosticar y optimizar rendimiento en pipelines Python y LangChain.

## 0. Setup tecnico

Instalamos solo `langchain-core` (no requiere API key). `langchain-openai` es **opcional**.

Toda la clase corre **sin claves**: simulamos el modelo con `RunnableLambda`.

In [ ]:
# Dependencia base (no requiere API key):
!pip install -q langchain-core

# Opcional, solo si quisieras usar OpenAI real (no es necesario para esta clase):
# !pip install -q langchain-openai

In [ ]:
import time
import asyncio
import cProfile
import pstats
import io
import re
from typing import List, Dict, Any

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel


def fake_llm_response(prompt_value):
    # Simulador de modelo: evita depender de APIs pagas.
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    return f"Respuesta simulada del modelo para:\n{text[:300]}"


fake_llm = RunnableLambda(fake_llm_response)
print("Setup listo. Modelo simulado disponible como fake_llm.")

## 1. Objetivo del notebook

Que puedas diagnosticar y optimizar rendimiento, y explicar: qué es profiling; qué mide `cProfile`; qué es un hotspot; `ncalls`, `tottime`, `cumtime`; CPU-bound vs I/O-bound; cuándo usar `asyncio`; ejecución async/batch en LangChain; y por qué hay que **volver a medir** después de optimizar.

## 2. Mapa visual: ciclo de profiling

```mermaid
flowchart LR
    A[Codigo lento] --> B[Ejecutar cProfile]
    B --> C[Leer tabla]
    C --> D[Detectar hotspot]
    D --> E[Optimizar]
    E --> F[Volver a medir]
    F --> G{Mejoro?}
    G -->|Si| H[Aceptar cambio]
    G -->|No| I[Revisar hipotesis]
    I --> B
```

## 3. Glosario

| Término | Explicación |
|---|---|
| Profiling | Medir dónde el programa consume tiempo. |
| Profiler | Herramienta que analiza rendimiento. |
| cProfile | Profiler incluido en Python. |
| Hotspot | Función que concentra mucho tiempo. |
| `ncalls` | Cantidad de llamadas a una función. |
| `tottime` | Tiempo propio de la función, sin subfunciones. |
| `cumtime` | Tiempo acumulado incluyendo subfunciones. |
| CPU-bound | Lentitud causada por cálculo. |
| I/O-bound | Lentitud causada por espera externa. |
| Asyncio | Librería de Python para concurrencia asíncrona. |
| Coroutine | Función async que puede pausarse y continuar. |
| `await` | Esperar sin bloquear completamente. |
| `gather` | Ejecutar varias tareas async juntas. |
| Concurrencia | Alternar tareas durante esperas. |
| Paralelismo | Ejecutar tareas realmente al mismo tiempo. |
| Batch | Procesar varios inputs juntos. |

## 4. Teoría + ejercicio real con cProfile

Medimos un código de limpieza de texto **lento**. Mirá `cumtime` para encontrar el hotspot.

In [ ]:
import cProfile, pstats, io, re

texts = ["Hola!!! Esto es una prueba 123..." * 1000 for _ in range(3000)]

def clean_text_slow(text):
    return ''.join(c.lower() for c in text if c.isalnum() or c.isspace())

def batch_process_slow(texts):
    return [clean_text_slow(t) for t in texts]

profiler = cProfile.Profile()
profiler.enable()
batch_process_slow(texts)
profiler.disable()

s = io.StringIO()
pstats.Stats(profiler, stream=s).sort_stats("cumtime").print_stats(10)
print(s.getvalue())

### Preguntas de interpretación

1. ¿Qué función consume más tiempo?
2. ¿Qué significa `cumtime`?
3. ¿Qué optimizarías primero?
4. ¿Qué error cometerías si optimizás otra función menos costosa?

## 5. Versión optimizada (y volver a medir)

Reemplazamos el bucle carácter por carácter por una **regex compilada**. Lo importante no es que “parezca mejor”, sino que **el tiempo baje medido con datos comparables**.

In [ ]:
def clean_text_fast(text):
    cleaned = re.sub(r'[^A-Za-z0-9\s]+', '', text)
    return cleaned.lower()

def batch_process_fast(texts):
    return [clean_text_fast(t) for t in texts]

profiler = cProfile.Profile()
profiler.enable()
batch_process_fast(texts)
profiler.disable()

s = io.StringIO()
pstats.Stats(profiler, stream=s).sort_stats("cumtime").print_stats(10)
print(s.getvalue())

## 6. Tabla: CPU-bound vs I/O-bound

| Operación | Tipo | Herramienta recomendada |
|---|---|---|
| Tokenizar millones de textos | CPU-bound | cProfile / optimización / multiprocessing |
| Ejecutar regex pesada | CPU-bound | profiling / optimización |
| Llamar API externa | I/O-bound | async / batching / timeout |
| Leer cloud storage | I/O-bound | async / cache |
| Guardar en base remota | I/O-bound | async / cola / batch |
| Esperar proveedor LLM | I/O-bound | async / batch / retry / timeout |

## 7. Gráfico Mermaid: ¿calcula o espera?

```mermaid
flowchart TB
    A[Sistema lento] --> B{Esta calculando o esperando?}
    B -->|Calculando| C[CPU-bound]
    B -->|Esperando red, disco o API| D[I/O-bound]
    C --> C1[cProfile]
    C --> C2[Optimizacion]
    C --> C3[Multiprocessing]
    D --> D1[Asyncio]
    D --> D2[Batching]
    D --> D3[Cache]
```

## 8. Mini ejemplo conceptual: secuencial vs async

Cinco descargas de 1 segundo cada una. Secuencial ≈ 5 s; async ≈ 1 s (las esperas se solapan). **Solo funciona porque el cuello de botella es I/O (esperar), no CPU.**

In [ ]:
import time

def descargar_doc(doc_id):
    time.sleep(1)
    return f"doc-{doc_id}"

inicio = time.time()
docs = [descargar_doc(i) for i in range(5)]
fin = time.time()
print("Documentos:", docs)
print("Tiempo total (secuencial):", round(fin - inicio, 2), "segundos")

In [ ]:
import asyncio, time

async def descargar_doc_async(doc_id):
    await asyncio.sleep(1)
    return f"doc-{doc_id}"

async def main():
    inicio = time.time()
    docs = await asyncio.gather(*[descargar_doc_async(i) for i in range(5)])
    fin = time.time()
    print("Documentos:", docs)
    print("Tiempo total (async):", round(fin - inicio, 2), "segundos")

# En Jupyter/Colab se usa await main() (NO asyncio.run()).
await main()

## 9. Código Python + LangChain: async con `ainvoke` / `abatch`

LangChain soporta ejecución asíncrona. Con `abatch` procesa varias llamadas async en paralelo lógico — ideal cuando el cuello de botella es la espera (API, LLM, storage).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
import asyncio, time

prompt = ChatPromptTemplate.from_template(
    "Resumi brevemente el siguiente documento:\n\n{documento}"
)

async def fake_async_model(prompt_value):
    await asyncio.sleep(1)
    text = prompt_value.to_string()
    return f"Resumen async simulado: {text[:120]}..."

async_chain = prompt | RunnableLambda(fake_async_model)

inputs = [{"documento": f"Documento numero {i}. " * 100} for i in range(5)]

inicio = time.time()
resultados = await async_chain.abatch(inputs)
fin = time.time()
print("Cantidad de resultados:", len(resultados))
print("Tiempo total (abatch):", round(fin - inicio, 2), "segundos")
print(resultados[0])

## 10. Ejercicio guiado — Clasificar operaciones

Decidí si cada operación es CPU-bound o I/O-bound y qué herramienta usar.

In [ ]:
operaciones = [
    {"operacion": "Tokenizar 1 millon de textos", "tipo": "CPU-bound"},
    {"operacion": "Consultar API externa de embeddings", "tipo": "I/O-bound"},
    {"operacion": "Leer archivos desde cloud storage", "tipo": "I/O-bound"},
    {"operacion": "Ejecutar regex pesada sobre textos enormes", "tipo": "CPU-bound"},
    {"operacion": "Guardar resultados en una base remota", "tipo": "I/O-bound"},
]
for op in operaciones:
    print(op["operacion"], "->", op["tipo"])

### Preguntas

1. ¿Qué operaciones conviene perfilar con `cProfile`?
2. ¿Qué operaciones conviene hacer async?
3. ¿Dónde puede servir cache?
4. ¿Dónde puede servir batching?

## 11. Errores comunes

- Usar async para código CPU-bound.
- Usar librerías bloqueantes dentro de funciones async.
- Lanzar demasiadas tareas sin límite.
- No usar timeouts ni manejar errores parciales.
- No comparar tiempos antes/después.
- Culpar al modelo sin medir el pipeline completo.
- No separar tiempo de preprocesamiento, inferencia e I/O.

## 12. Cierre conceptual

Primero **medí** (cProfile), después decidí: si el problema es **cálculo** (CPU-bound), optimizás el código; si es **espera** (I/O-bound), usás `asyncio`/`abatch`. Y siempre **volvé a medir**: la mejora se demuestra con números comparables, no con intuición.